# llm-hpo-optimizer — Google Colab Quickstart

This notebook shows three use cases:
1. **Custom adapter** (zero extra dependencies — runs on CPU instantly)
2. **Bayesian search** with Optuna
3. **nanoGPT adapter** (requires GPU + dataset)

Start with **Section 1** to install and verify everything works.

## 1. Installation

**Option A — install from GitHub (recommended once repo is public):**

In [ ]:
# Option A: Install directly from GitHub
# Replace YOUR_USERNAME with your actual GitHub username
# !pip install git+https://github.com/YOUR_USERNAME/llm-hpo-optimizer.git -q

# Option B: Upload the project zip and install from it
# 1. Zip your project folder locally
# 2. Upload via the Files panel on the left (folder icon)
# 3. Run the cells below

# Unzip if you uploaded a zip
# !unzip llm-hpo-optimizer.zip -d llm-hpo-optimizer

# Option C: Mount Google Drive (if project is saved there)
# from google.colab import drive
# drive.mount('/content/drive')
# !pip install -e /content/drive/MyDrive/llm-hpo-optimizer -q

# --- CURRENTLY ACTIVE: install from uploaded/cloned folder ---
!pip install -e ./llm-hpo-optimizer -q

print('Installation complete!')

In [ ]:
# Verify installation
import llm_hpo_optimizer
print(f'llm-hpo-optimizer version: {llm_hpo_optimizer.__version__}')

## 2. Custom Adapter + Random Search (No Extra Dependencies)

This works on any Colab runtime — no GPU needed.

In [ ]:
import math
import time

from llm_hpo_optimizer import (
    Categorical,
    LogUniformFloat,
    SearchSpace,
    TrainingAdapter,
    TrainingResult,
    UniformFloat,
    UniformInt,
    optimize,
)


class SyntheticAdapter(TrainingAdapter):
    """Toy adapter with a synthetic loss landscape.
    Optimal: lr≈3e-4, hidden_size=256, dropout=0.1, num_layers=4
    """
    def run_trial(self, hyperparams: dict, seed: int) -> TrainingResult:
        t0 = time.time()
        lr      = hyperparams['learning_rate']
        hidden  = hyperparams['hidden_size']
        dropout = hyperparams['dropout']
        layers  = hyperparams['num_layers']

        val_loss = (
            0.3
            + (math.log(lr / 3e-4)) ** 2 * 0.5
            + ((hidden - 256) / 256) ** 2 * 0.2
            + (dropout - 0.1) ** 2 * 0.5
            + ((layers - 4) / 4) ** 2 * 0.1
        )
        elapsed = time.time() - t0 + 1e-4

        return TrainingResult(
            train_loss=val_loss * 0.9,
            val_loss=val_loss,
            best_val_loss=val_loss,
            best_val_iter=50,
            training_iterations=100,
            parameter_count=hidden * layers * 4,
            model_size_mb=hidden * layers * 4 * 4 / (1024 ** 2),
            peak_memory_mb=hidden * 0.1,
            training_time_seconds=elapsed,
            samples_seen=10_000,
            samples_per_second=10_000 / elapsed,
            device='cpu',
            seed=seed,
        )


search_space = SearchSpace({
    'learning_rate': LogUniformFloat(1e-5, 1e-2),
    'hidden_size':   Categorical([64, 128, 256, 512]),
    'dropout':       UniformFloat(0.0, 0.5),
    'num_layers':    UniformInt(1, 8),      # new integer type
})
baseline = {'learning_rate': 1e-3, 'hidden_size': 128, 'dropout': 0.2, 'num_layers': 2}

result = optimize(
    method='random',
    adapter=SyntheticAdapter(),
    search_space=search_space,
    baseline_hyperparams=baseline,
    optimization_iterations=15,
    seed=42,
    verbose=True,           # prints progress per trial
    output_path='results.json',
)

print(f"\nBaseline:          {result['baseline_config']}")
print(f"Best (by loss):    {result['best_loss_config']}")
print(f"  val_loss: {result['best_loss_value']:.4f}")
print(f"Best (balanced):   {result['best_balanced_config']}")
print(f"  score:    {result['best_balanced_score']:.4f}")
print(f"Pareto front size: {len(result['pareto_front'])}")
print(f"Total trials:      {len(result['all_records'])}")

## 3. Bayesian Search with Optuna (Better Results, Same Code)

Just switch `method='optuna'`. Optuna learns from each trial to suggest better configs.

In [ ]:
# Install Optuna (only needed once per session)
!pip install optuna -q
print('Optuna ready!')

In [ ]:
result_bayes = optimize(
    method='optuna',           # <-- only change from above
    adapter=SyntheticAdapter(),
    search_space=search_space,
    baseline_hyperparams=baseline,
    optimization_iterations=15,
    seed=42,
    verbose=True,
)

print(f"\nBest (by loss):  {result_bayes['best_loss_config']}")
print(f"  val_loss: {result_bayes['best_loss_value']:.4f}")

## 4. Inspect Results

In [ ]:
import json

# All trial records
for rec in result['all_records']:
    tag = 'BASELINE' if rec['is_baseline'] else f"score={rec['score']:.3f}" if rec['score'] is not None else 'FAILED'
    print(f"  Trial {rec['experiment_id']:>2} | val_loss={rec['metrics']['best_val_loss']:.4f} | {tag} | {rec['hyperparameters']}")

In [ ]:
# Pareto-optimal configs
print('Pareto front (non-dominated configs):')
for p in result['pareto_front']:
    print(f"  val_loss={p['metrics']['best_val_loss']:.4f} | mem={p['resources']['peak_memory_mb']:.1f} MB | {p['hyperparameters']}")

In [ ]:
# Load saved JSON results from disk
with open('results.json') as f:
    saved = json.load(f)
print(f"Loaded {len(saved['all_records'])} records from results.json")

## 5. nanoGPT Adapter (Requires GPU + Dataset)

> **Runtime:** Make sure you have selected **GPU** in *Runtime → Change runtime type*.

This section trains a real nanoGPT model on the tiny Shakespeare dataset.

In [ ]:
# Install torch extras for the nanoGPT adapter
!pip install "llm-hpo-optimizer[nanogpt]" -q   # or: pip install torch numpy -q

import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# 1. Clone nanoGPT and make it importable
!git clone https://github.com/karpathy/nanoGPT.git -q
import sys
sys.path.insert(0, '/content/nanoGPT')

# 2. Prepare the tiny Shakespeare dataset
!python /content/nanoGPT/data/shakespeare_char/prepare.py

print('nanoGPT and dataset ready!')

In [ ]:
from llm_hpo_optimizer import optimize

result_gpt = optimize(
    dataset_dir='/content/nanoGPT/data/shakespeare_char',
    method='optuna',
    optimization_iterations=10,
    training_iterations_per_trial=500,   # keep short for demo
    seed=42,
    verbose=True,
    output_path='nanogpt_results.json',
)

print(f"\nBest config (balanced): {result_gpt['best_balanced_config']}")
print(f"Best val_loss:          {result_gpt['best_loss_value']:.4f}")
print(f"Pareto front size:      {len(result_gpt['pareto_front'])}")

## 6. Custom Scoring Weights

Tune how much each objective matters to the balanced score.

In [ ]:
# Optimise heavily for low memory (e.g. deployment on edge device)
result_mem = optimize(
    method='random',
    adapter=SyntheticAdapter(),
    search_space=search_space,
    baseline_hyperparams=baseline,
    optimization_iterations=15,
    seed=42,
    score_weights={
        'val_loss':        0.4,
        'peak_memory_mb':  0.4,   # heavily penalise memory
        'parameter_count': 0.1,
        'training_time':   0.1,
    },
)

print(f"Memory-optimised best config: {result_mem['best_balanced_config']}")

## 7. Save Results to Google Drive (Optional)

In [ ]:
# Mount Google Drive and copy results there
# from google.colab import drive
# drive.mount('/content/drive')
# !cp results.json /content/drive/MyDrive/hpo_results.json
# print('Saved to Google Drive!')

# OR: Download directly to your computer
from google.colab import files
files.download('results.json')